# Descriptive analytics with pandas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/data-analysis/descriptive-analytics.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/data-analysis/descriptive-analytics.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup

The notebook checks its required libraries and installs only packages that are missing in Colab or Binder. For local Jupyter use, see the [environment guidance](https://github.com/gromicho/teaching/blob/main/docs/SETUP.md).


## From observations to questions
The small fruit table has been used in the teaching collection. Before analysing it, inspect the units, column names and possible outliers. A summary describes the available observations; it does not explain their cause or guarantee that they represent a wider population.

The earlier COVID-19 demonstration is preserved in the instructor archive. This maintained first example uses the bundled small table so that learning pandas does not depend on an external service or imply a current epidemiological forecast.


In [ ]:
# Use installed packages, install only missing ones, without version pins.
from importlib.util import find_spec
import subprocess
import sys

required_packages = {
    'matplotlib': 'matplotlib',
    'pandas': 'pandas',
}
missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if find_spec(import_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

# Prefer the checked-in file locally; Colab downloads the same frozen edition.
data_path = next((p for p in [Path("data/fruits.csv"), Path("fruits.csv")]
                 if p.is_file()), Path("fruits.csv"))
if not data_path.is_file():
    url = "https://raw.githubusercontent.com/gromicho/teaching/main/data/fruits.csv"
    with urlopen(url, timeout=45) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22":
        raise ValueError("Dataset checksum mismatch; do not use an unverified copy.")
    data_path.write_bytes(payload)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22", "Unexpected local data version"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
fruits = pd.read_csv(data_path, sep=';', decimal=',')
print(fruits.shape)
print(fruits.columns.tolist())
fruits.head()


## Inspect before cleaning
Which columns are numerical? What does one row represent? Compare the smallest and largest values. Do not silently delete an unusual observation merely because it looks inconvenient.


In [ ]:
print(fruits.dtypes)
print(fruits.isna().sum())
fruits.describe(include='all')


In [ ]:
numeric = fruits.select_dtypes('number')
assert len(numeric) > 0
numeric.hist(figsize=(10,5))
plt.tight_layout()
plt.show()


## Group and compare
Choose a categorical column and compare numerical summaries across its groups. Are the groups equally large? Could a few observations dominate their means?


In [ ]:
categories = fruits.select_dtypes(exclude='number').columns
if len(categories):
    group = categories[0]
    print(fruits.groupby(group).size())
    display(fruits.groupby(group)[list(numeric.columns)].agg(['count','mean','median']))


## Check your interpretation
Write three statements supported directly by the table, and one question the table cannot answer. Keep descriptive statements separate from explanations and predictions.
